In [1]:
!pip install obspy
!pip install numpy pandas matplotlib
!pip install scipy
!pip install keras
!pip install tensorflow
!pip install wurlitzer

import os
import obspy.clients.fdsn
import numpy as np
import pandas as pd
import obspy
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from datetime import datetime
import scipy
import warnings
warnings.filterwarnings("ignore")


import pylab as plt
import warnings
import os
import h5py
import matplotlib as mpl
%matplotlib inline

import sklearn
from sklearn import linear_model

import pandas as pd
import seaborn as sns

import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPooling2D, LSTM
from keras import losses
from keras.utils import to_categorical
from keras.layers import BatchNormalization
import keras.backend as K

from __future__ import print_function

from collections import defaultdict
import pickle
from PIL import Image

from six.moves import range

from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Embedding, Dropout, LeakyReLU
from keras.layers import Conv2D, MaxPooling2D
from keras.models import Model
from keras.optimizers import Adam
from keras.utils import Progbar
import numpy as np

from numpy import zeros
from numpy import ones
from numpy import expand_dims
from numpy.random import randn
from numpy.random import randint
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers import Activation
from keras.layers import Concatenate
from keras.initializers import RandomNormal
from matplotlib import pyplot

np.random.seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 67.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.25
    Uninstalling SQLAlchemy-2.0.25:
      Successfully uninstalled SQLAlchemy-2.0.25
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.53 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: keras
    Found existing installation: keras 3.3.3
    Uninstalling keras-3.3.3:
      Successfully uninstalled keras-3.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the followi

2024-08-02 10:03:46.678366: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-02 10:03:46.678475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-02 10:03:46.826962: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import os
os.getcwd()

'/kaggle/working'

In [3]:
import numpy as np
import os

# Define the folder path
folder_path = '/kaggle/input/dataset_GAN'

# Load numpy arrays from the specified folder
W = np.load(os.path.join(folder_path, '/kaggle/input/dataset-gan/Dataset_waveforms_train.npy'))
Y = np.load(os.path.join(folder_path, '/kaggle/input/dataset-gan/Dataset_labels_train.npy'))

print(W.shape, Y.shape)


(1898, 3, 12000) (1898, 1)


In [4]:
W.shape, Y.shape

((1898, 3, 12000), (1898, 1))

In [5]:
W = W.reshape(-1,12000,3,1)
W.shape

(1898, 12000, 3, 1)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assuming W and Y are your data and label arrays
combined = list(zip(W, Y))
np.random.shuffle(combined)
W_shuffled, Y_shuffled = zip(*combined)

# Convert back to numpy arrays
W_shuffled = np.array(W_shuffled)
Y_shuffled = np.array(Y_shuffled)

In [ ]:
# First, split into train+val and test
W_trainval, W_test, Y_trainval, Y_test = train_test_split(W_shuffled, Y_shuffled, test_size=0.2, random_state=42)

# Then split train+val into train and val
W_train, W_val, Y_train, Y_val = train_test_split(W_trainval, Y_trainval, test_size=0.2, random_state=42)

# Check the shapes
print("W_train shape:", W_train.shape)
print("Y_train shape:", Y_train.shape)
print("W_val shape:", W_val.shape)
print("Y_val shape:", Y_val.shape)
print("W_test shape:", W_test.shape)
print("Y_test shape:", Y_test.shape)

In [ ]:
print("Train label distribution:", np.bincount(Y_train.flatten().astype(int)))
print("Validation label distribution:", np.bincount(Y_val.flatten().astype(int)))
print("Test label distribution:", np.bincount(Y_test.flatten().astype(int)))

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Concatenate, Conv2D, BatchNormalization, Activation, Embedding, Flatten, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from collections import defaultdict
from tensorflow.keras.utils import Progbar
from tensorflow.keras.layers import MaxPooling2D

def build_generator(latent_size):
    in_label = Input(shape=(1,))
    li = Embedding(100, latent_size)(in_label)
    n_nodes = 100 * 3
    li = Dense(n_nodes)(li)
    li = Reshape((100, 3, 1))(li)

    in_lat = Input(shape=(latent_size,))
    n_nodes = 100 * 3 * 384
    gen = Dense(n_nodes)(in_lat)
    gen = Activation('relu')(gen)
    gen = Reshape((100, 3, 384))(gen)

    merge = Concatenate()([gen, li])

    gen = Conv2D(32, (5, 5), activation='relu', padding='SAME')(merge)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((2, 1))(gen)  # 200 x 3 x 32
    gen = Conv2D(64, (5, 5), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((2, 1))(gen)  # 400 x 3 x 64
    gen = Conv2D(128, (3, 3), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((2, 1))(gen)  # 800 x 3 x 128
    gen = Conv2D(256, (3, 3), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((2, 1))(gen)  # 1600 x 3 x 256
    gen = Conv2D(128, (3, 3), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((3, 1))(gen)  # 4800 x 3 x 128
    gen = Conv2D(64, (3, 3), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = UpSampling2D((5, 1))(gen)  # 12000 x 3 x 64 (approximation)
    gen = BatchNormalization()(gen)
    gen = MaxPooling2D(pool_size=(2, 1))(gen)  # Downsample by factor of 2
    gen = Conv2D(64, (3, 3), activation='relu', padding='SAME')(gen)
    gen = BatchNormalization()(gen)
    gen = Conv2D(1, (3, 3), activation='tanh', padding='SAME')(gen)

    out_layer = Reshape((12000, 3, 1))(gen)

    model = Model([in_lat, in_label], out_layer)
    print("------------------Generator Model--------------------")
    print(model.summary())
    return model

In [ ]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.constraints import Constraint
import tensorflow as tf

class SpectralNormalization(Constraint):
    def __init__(self, iterations=1):
        self.iterations = iterations

    def __call__(self, W):
        w_shape = W.shape.as_list()
        W = tf.reshape(W, [-1, w_shape[-1]])

        u = tf.random.normal([1, w_shape[-1]])
        
        for _ in range(self.iterations):
            v = tf.matmul(u, tf.transpose(W))
            v = v / tf.norm(v)
            u = tf.matmul(tf.transpose(W), v)
            u = u / tf.norm(u)
            
        sigma = tf.matmul(tf.matmul(u, tf.transpose(W)), v)
        w_norm = W / sigma
        return tf.reshape(w_norm, w_shape)



In [ ]:
def build_discriminator():
    in_image = Input(shape=(12000, 3, 1))
    fe = Conv2D(32, (5, 5), padding='same', activation='relu', kernel_constraint=SpectralNormalization())(in_image)
    fe = BatchNormalization()(fe)
    fe = Conv2D(64, (5, 5), padding='same', activation='relu')(fe)
    fe = BatchNormalization()(fe)
    fe = Conv2D(128, (3, 3), padding='same', activation='relu')(fe)
    fe = BatchNormalization()(fe)
    fe = Conv2D(256, (3, 3), padding='same', activation='relu')(fe)
    fe = BatchNormalization()(fe)
    fe = Conv2D(128, (3, 3), padding='same', activation='relu')(fe)
    fe = BatchNormalization()(fe)
    fe = Conv2D(64, (3, 3), padding='same', activation='relu')(fe)
    fe = Flatten()(fe)
    out1 = Dense(1, activation='sigmoid')(fe)
    out2 = Dense(3, activation='softmax')(fe)
    model = Model(in_image, [out1, out2])

    print("------------------Discriminator Model--------------------")
    print(model.summary())
    return model

In [ ]:
def validation_step(X_val, y_val):
    noise = tf.random.normal([len(X_val), latent_size])
    generated_images = generator.predict([noise, y_val], verbose=0)

    real_output, real_aux = discriminator.predict(X_val, verbose=0)
    fake_output, fake_aux = discriminator.predict(generated_images, verbose=0)

    # Update this line to use the correct number of classes
    y_val_onehot = tf.one_hot(y_val, depth=n_classes)
    
    # Print shapes
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    print("generated_images shape:", generated_images.shape)
    print("real_output shape:", real_output.shape)
    print("real_aux shape:", real_aux.shape)
    print("fake_output shape:", fake_output.shape)
    print("fake_aux shape:", fake_aux.shape)
    
    # Check for NaN values
    print("Generated images NaN check:", tf.math.reduce_any(tf.math.is_nan(generated_images)))
    print("Real output NaN check:", tf.math.reduce_any(tf.math.is_nan(real_output)))
    print("Fake output NaN check:", tf.math.reduce_any(tf.math.is_nan(fake_output)))
    print("Real aux NaN check:", tf.math.reduce_any(tf.math.is_nan(real_aux)))
    print("Fake aux NaN check:", tf.math.reduce_any(tf.math.is_nan(fake_aux)))
    
    # Ensure correct shapes
    y_val = tf.squeeze(y_val)  # Remove extra dimensions if any
    real_aux = tf.squeeze(real_aux)  # Remove extra dimensions if any
    fake_aux = tf.squeeze(fake_aux)  # Remove extra dimensions if any

    # 1. Check auxiliary classifier output
    print("Real aux min/max:", tf.reduce_min(real_aux).numpy(), tf.reduce_max(real_aux).numpy())
    print("Fake aux min/max:", tf.reduce_min(fake_aux).numpy(), tf.reduce_max(fake_aux).numpy())

    # 2. Verify the labels
    print("y_val min/max:", tf.reduce_min(y_val).numpy(), tf.reduce_max(y_val).numpy())
    print("y_val unique values:", np.unique(y_val))

    # 6. Check for any extreme values in the auxiliary outputs
    print("Real aux contains infinity:", tf.math.reduce_any(tf.math.is_inf(real_aux)))
    print("Fake aux contains infinity:", tf.math.reduce_any(tf.math.is_inf(fake_aux)))

    # Compute losses
    disc_real_loss = tf.keras.losses.binary_crossentropy(tf.ones_like(real_output), real_output)
    disc_fake_loss = tf.keras.losses.binary_crossentropy(tf.zeros_like(fake_output), fake_output)

    # 3. Use categorical_crossentropy for debugging
    y_val_onehot = tf.one_hot(tf.cast(y_val, tf.int32), depth=n_classes)
    
    disc_aux_real_loss_cat = tf.keras.losses.categorical_crossentropy(y_val_onehot, real_aux)
    disc_aux_fake_loss_cat = tf.keras.losses.categorical_crossentropy(y_val_onehot, fake_aux)
   
    # 5. Add epsilon to prevent log(0)
    epsilon = 1e-7
    disc_aux_real_loss = tf.keras.losses.sparse_categorical_crossentropy(y_val, real_aux + epsilon)
    disc_aux_fake_loss = tf.keras.losses.sparse_categorical_crossentropy(y_val, fake_aux + epsilon)

    gen_loss = tf.keras.losses.binary_crossentropy(tf.ones_like(fake_output), fake_output)
    
    # Check individual losses
    print("Disc real loss:", tf.math.reduce_mean(disc_real_loss).numpy())
    print("Disc fake loss:", tf.math.reduce_mean(disc_fake_loss).numpy())
    print("Disc aux real loss (sparse):", tf.math.reduce_mean(disc_aux_real_loss).numpy())
    print("Disc aux fake loss (sparse):", tf.math.reduce_mean(disc_aux_fake_loss).numpy())
    print("Disc aux real loss (categorical):", tf.math.reduce_mean(disc_aux_real_loss_cat).numpy())
    print("Disc aux fake loss (categorical):", tf.math.reduce_mean(disc_aux_fake_loss_cat).numpy())
    print("Gen loss:", tf.math.reduce_mean(gen_loss).numpy())

    return [tf.math.reduce_mean(disc_real_loss + disc_fake_loss).numpy(),
            tf.math.reduce_mean(disc_aux_real_loss).numpy(),
            tf.math.reduce_mean(disc_aux_fake_loss).numpy()], \
           [tf.math.reduce_mean(gen_loss).numpy(),
            tf.math.reduce_mean(disc_aux_fake_loss).numpy()]


 


In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam


if __name__ == '__main__':
    
    # Parameters
    nb_epochs = 20
    batch_size = 16
    latent_size = 100
    n_classes = 3

    adam_lr = 0.0000001
    adam_beta_1 = 0.5


    # Build and compile the generator
    generator = build_generator(latent_size)
    generator.compile(optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
                      loss='binary_crossentropy')

    # Build and compile the discriminator
    discriminator = build_discriminator()
    discriminator.compile(
        optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
        loss=['binary_crossentropy', 'sparse_categorical_crossentropy']
    )

    # Build the combined model
    latent = Input(shape=(latent_size,))
    image_class = Input(shape=(1,), dtype='int32')
    fake = generator([latent, image_class])
    discriminator.trainable = False
    fake_output, aux = discriminator(fake)
    combined = Model([latent, image_class], [fake_output, aux])
    combined.compile(
        optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
        loss=['binary_crossentropy', 'sparse_categorical_crossentropy']
    )

    # Compile combined model
    combined.compile(
        optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
        loss=['binary_crossentropy', 'sparse_categorical_crossentropy']
    )

    # Assuming you have already defined build_generator and build_discriminator

    # Set up optimizers
    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=adam_lr, beta_1=adam_beta_1)
    discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=adam_lr, beta_1=adam_beta_1)

    # Call build with the trainable variables
    generator_optimizer.build(generator.trainable_variables)
    discriminator_optimizer.build(discriminator.trainable_variables)

    # Define loss functions
    binary_cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
    categorical_cross_entropy = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

    def generator_loss(fake_output):
        return binary_cross_entropy(tf.ones_like(fake_output), fake_output)
    
    
    def discriminator_loss(real_output, fake_output, aux_real, aux_fake, real_labels_smooth):
        real_loss = binary_cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
        fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
    
        # Round the smoothed labels back to integers if necessary
        real_labels_int = tf.cast(tf.round(real_labels_smooth), tf.int32)

        aux_loss_real = categorical_cross_entropy(real_labels_int, aux_real)
        aux_loss_fake = categorical_cross_entropy(real_labels_int, aux_fake)
        total_loss = real_loss + fake_loss + aux_loss_real + aux_loss_fake
        return total_loss
  
    
    @tf.function
    def train_generator_step(noise, labels):
        with tf.GradientTape() as tape:
            generated_images = generator([noise, labels], training=True)
            fake_output, _ = discriminator(generated_images, training=False)
            gen_loss = generator_loss(fake_output)

        gradients = tape.gradient(gen_loss, generator.trainable_variables)
        generator_optimizer.apply_gradients(zip(gradients, generator.trainable_variables))
        return gen_loss

    def gradient_penalty(real_images, fake_images):
        batch_size = tf.shape(real_images)[0]
        alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
        diff = fake_images - real_images
        interpolated = real_images + alpha * diff

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = discriminator(interpolated, training=True)[0]

        grads = gp_tape.gradient(pred, [interpolated])[0]
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]))
        gp = tf.reduce_mean((norm - 1.0) ** 2)
        return gp

    @tf.function
    def train_discriminator_step(real_images, real_labels, noise, fake_labels):
        with tf.GradientTape() as tape:
            generated_images = generator([noise, fake_labels], training=False)

            # Ensure the batch sizes match
            batch_size = tf.shape(real_images)[0]
            generated_images = generated_images[:batch_size]
            fake_labels = fake_labels[:batch_size]

            real_output, aux_real = discriminator(real_images, training=True)
            fake_output, aux_fake = discriminator(generated_images, training=True)

            # Convert labels to float32 before adding noise
            real_labels_float = tf.cast(real_labels, tf.float32)
            fake_labels_float = tf.cast(fake_labels, tf.float32)

            # Add noise to the labels
            real_labels_smooth = real_labels_float + tf.random.uniform(tf.shape(real_labels_float), 0, 0.1)
            fake_labels_smooth = fake_labels_float + tf.random.uniform(tf.shape(fake_labels_float), 0, 0.1)

            disc_loss = discriminator_loss(real_output, fake_output, aux_real, aux_fake, real_labels_smooth)
            gp = gradient_penalty(real_images, generated_images)
            disc_loss += 10 * gp  # Lambda = 10

        gradients = tape.gradient(disc_loss, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(gradients, discriminator.trainable_variables))
        return disc_loss

    # Check Loss Functions
    dummy_noise = tf.random.normal((batch_size, latent_size))
    dummy_labels = tf.random.uniform((batch_size,), minval=0, maxval=n_classes, dtype=tf.int32)
    dummy_images = tf.random.normal((batch_size, 28, 28, 1))  # Adjust the shape as per your input shape
    dummy_real_output = tf.random.normal((batch_size, 1))
    dummy_fake_output = tf.random.normal((batch_size, 1))
    dummy_aux_real = tf.random.uniform((batch_size, n_classes))
    dummy_aux_fake = tf.random.uniform((batch_size, n_classes))

    try:
        print("Checking generator loss:")
        gen_loss_check = generator_loss(dummy_fake_output)
        print("Generator loss:", gen_loss_check.numpy())

        print("Checking discriminator loss:")
        disc_loss_check = discriminator_loss(dummy_real_output, dummy_fake_output, dummy_aux_real, dummy_aux_fake, dummy_labels)
        print("Discriminator loss:", disc_loss_check.numpy())

        print("Loss functions are working correctly.")

    except Exception as e:
        print("There is an issue with the loss functions:", e)
        raise


    # Load and preprocess data
    # Replace your current data loading with these shuffled and split datasets
    X_train, X_val, X_test = W_train, W_val, W_test
    y_train, y_val, y_test = Y_train, Y_val, Y_test

    nb_train, nb_val, nb_test = X_train.shape[0], X_val.shape[0], X_test.shape[0]
    
    import numpy as np

    assert not np.any(np.isnan(X_train)), "X_train contains NaN values"
    assert not np.any(np.isnan(y_train)), "y_train contains NaN values"


    nb_train, nb_val, nb_test = X_train.shape[0], X_val.shape[0], X_test.shape[0]

    train_history = {'generator': [], 'discriminator': []}
    val_history = {'generator': [], 'discriminator': []}
    test_history = {'generator': [], 'discriminator': []}

    best_val_loss = float('inf')
    best_epoch = 0

    # Convert your data to TensorFlow tensors
    X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)    
    
    print("Checking for NaN or infinite values in training data:")
    print("X_train:", np.isnan(X_train).any(), np.isinf(X_train).any())
    print("y_train:", np.isnan(y_train).any(), np.isinf(y_train).any())

    # Add gradient clipping to your optimizers
    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=adam_lr, beta_1=adam_beta_1, clipvalue=1.0)
    discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=adam_lr, beta_1=adam_beta_1, clipvalue=1.0)

    # Training loop
    for epoch in range(nb_epochs):
        print(f'Epoch {epoch + 1} of {nb_epochs}')
        nb_batches = int(np.ceil(nb_train / float(batch_size)))
        progress_bar = tf.keras.utils.Progbar(target=nb_batches)

        epoch_gen_loss = []
        epoch_disc_loss = []

        for index in range(nb_batches):

            progress_bar.update(index)

            # Get a batch of real images
            start_idx = index * batch_size
            end_idx = min((index + 1) * batch_size, nb_train)
            current_batch_size = end_idx - start_idx

            image_batch = X_train[start_idx:end_idx]
            label_batch = y_train[start_idx:end_idx]

            # Generate fake images
            noise = tf.random.normal([current_batch_size, latent_size])
            sampled_labels = tf.random.uniform([current_batch_size], minval=0, maxval=n_classes, dtype=tf.int32)

            generated_images = generator([noise, sampled_labels], training=False)

            # Train discriminator
            disc_loss = train_discriminator_step(image_batch, label_batch, noise, sampled_labels)
            epoch_disc_loss.append(disc_loss)

            # Train generator
            noise = tf.random.normal([current_batch_size, latent_size])
            sampled_labels = tf.random.uniform([current_batch_size], minval=0, maxval=n_classes, dtype=tf.int32)
            gen_loss = train_generator_step(noise, sampled_labels)
            epoch_gen_loss.append(gen_loss)

        # Compute average losses for this epoch
        train_history['discriminator'].append(np.mean(epoch_disc_loss, axis=0))
        train_history['generator'].append(np.mean(epoch_gen_loss, axis=0))


        # Validation step
        
        print(f'\nValidation for epoch {epoch + 1}:')
        disc_val_loss, gen_val_loss = validation_step(X_val, y_val)

        # Record validation results
        val_history['discriminator'].append(disc_val_loss)
        val_history['generator'].append(gen_val_loss)

        print(f"Validation Discriminator Loss: {disc_val_loss}")
        print(f"Validation Generator Loss: {gen_val_loss}")

        noise = np.random.uniform(-1, 1, (nb_val, latent_size))
        sampled_labels = np.random.randint(0, n_classes, nb_val)
        generated_images = generator.predict([noise, sampled_labels.reshape((-1, 1))], verbose=0)
        
        
        # Use X_val instead of image_batch
        X = np.concatenate([X_val, generated_images])
        y = np.array([1] * nb_val + [0] * nb_val)
        aux_y = np.concatenate([y_val, sampled_labels.reshape((-1, 1))])

        

        # Evaluate discriminator
        disc_val_loss = discriminator.evaluate(X, [y, aux_y], verbose=False)

        # Evaluate generator
        noise = np.random.uniform(-1, 1, (2 * nb_val, latent_size))
        sampled_labels = np.random.randint(0, n_classes, 2 * nb_val)
        trick = np.ones(2 * nb_val)
        gen_val_loss = combined.evaluate(
            [noise, sampled_labels.reshape((-1, 1))],
            [trick, sampled_labels],
            verbose=False
        )

        val_history['discriminator'].append(disc_val_loss)
        val_history['generator'].append(gen_val_loss)
        
        # In your training loop
        print(f'\nValidation for epoch {epoch + 1}:')
        disc_val_loss, gen_val_loss = validation_step(X_val, y_val)
        print(f"Validation Discriminator Loss: {disc_val_loss}")
        print(f"Validation Generator Loss: {gen_val_loss}")

        # Check if this is the best model so far
        current_val_loss = disc_val_loss[0] + gen_val_loss[0]  # You might want to adjust this metric
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_epoch = epoch
            # Save the best model
            generator.save_weights(f'/kaggle/working/best_generator.hdf5')
            discriminator.save_weights(f'/kaggle/working/best_discriminator.hdf5')

        print(f"Epoch {epoch+1}/{nb_epochs} completed. Current best epoch: {best_epoch+1}")

    print(f"Training completed. Best epoch was {best_epoch+1}")

    # Final test on the best model
    generator.load_weights('/kaggle/working/best_generator.hdf5')
    discriminator.load_weights('/kaggle/working/best_discriminator.hdf5')

    noise = np.random.uniform(-1, 1, (nb_test, latent_size))
    sampled_labels = np.random.randint(0, n_classes, nb_test)
    generated_images = generator.predict([noise, sampled_labels.reshape((-1, 1))], verbose=0)

    X = np.concatenate([X_test, generated_images])
    y = np.array([1] * nb_test + [0] * nb_test)
    aux_y = np.concatenate([y_test, sampled_labels.reshape((-1,1))])

    trick = np.ones(nb_test)
    final_disc_loss = discriminator.evaluate(X, [y, aux_y], verbose=False)
    final_gen_loss = combined.evaluate(
        [noise, sampled_labels.reshape((-1, 1))],
        [trick, sampled_labels],
        verbose=False
    )

    print(f"Final Test Discriminator Loss: {final_disc_loss}")
    print(f"Final Test Generator Loss: {final_gen_loss}")

    # Save the training history for later analysis
    np.save('/kaggle/working/train_history.npy', train_history)
    np.save('/kaggle/working/val_history.npy', val_history)
    np.save('/kaggle/working/test_history.npy', test_history)
    
    

In [ ]:
# After the training loop
print("Performing final test...")
disc_test_loss, gen_test_loss = validation_step(X_test, y_test)

# Record test results
test_history['discriminator'].append(disc_test_loss)
test_history['generator'].append(gen_test_loss)

print(f"Final Test Discriminator Loss: {disc_test_loss}")
print(f"Final Test Generator Loss: {gen_test_loss}")

In [ ]:
    # After training, create plotting lists
    gen_train = train_history['generator']
    dis_train = train_history['discriminator']
    gen_test = test_history['generator']
    dis_test = test_history['discriminator']

    # Now you can plot these
    plt.figure(figsize=(10, 6))
    plt.plot(gen_test, label='Generator (Test)')
    plt.plot(gen_train, label='Generator (Train)')
    plt.plot(dis_train, label='Discriminator (Train)')
    plt.plot(dis_test, label='Discriminator (Test)')
    plt.grid()
    plt.legend()
    plt.title('Loss per Epoch')
    plt.xlabel('Epoch')
    plt.ylabel(r'$L_c$')
    plt.show()


In [ ]:
# Load the best weights
generator.load_weights('/kaggle/working/best_generator.hdf5')

# Generate images
n_samples = 100
noise = np.random.uniform(-1, 1, (n_samples, latent_size))
sampled_labels = np.random.randint(0, 3, n_samples)  # 0: earthquake, 1: non-earthquake, 2: rockfall
generated_waveforms = generator.predict([noise, sampled_labels.reshape((-1, 1))])

# Now you can analyze or visualize these generated waveforms

In [ ]:
generated_images.shape, W.shape

In [ ]:
plt.figure(figsize=(15, 5))
for i in range(3):
    plt.plot(generated_images[4, i, :], label=f'Channel {i+1}')
plt.legend()
plt.title('All channels of the 10th generated image')
plt.show()

In [ ]:
gen_train = []
dis_train = []
gen_test = []
dis_test = []

for i in range(len(train_history['generator'])):
    gen_train.append(train_history['generator'][i])
    dis_train.append(train_history['discriminator'][i])

for i in range(len(test_history['generator'])):
    gen_test.append(test_history['generator'][i])
    dis_test.append(test_history['discriminator'][i])


In [ ]:
# Debug prints
print("train_history['generator']: ", train_history['generator'])
print("train_history['discriminator']: ", train_history['discriminator'])
print("test_history['generator']: ", test_history['generator'])
print("test_history['discriminator']: ", test_history['discriminator'])


In [ ]:
gen_train=[]
dis_train=[]
gen_test=[]
dis_test=[]

for i in range(len(train_history['generator'])):
    gen_train.append(train_history['generator'][i][0])
    dis_train.append(train_history['discriminator'][i][0])

for i in range(len(test_history['generator'])):
    gen_test.append(test_history['generator'][i][0])
    dis_test.append(test_history['discriminator'][i][0])

In [ ]:
print("train_history keys:", train_history.keys())
print("test_history keys:", test_history.keys())

print("Length of train_history['generator']:", len(train_history['generator']))
print("Length of train_history['discriminator']:", len(train_history['discriminator']))
print("Length of test_history['generator']:", len(test_history['generator']))
print("Length of test_history['discriminator']:", len(test_history['discriminator']))

In [ ]:
print("gen_test:", gen_test)
print("gen_train:", gen_train)
print("dis_train:", dis_train)
print("dis_test:", dis_test)

In [ ]:
plt.plot(gen_test,label='Generator (Test)')
plt.plot(gen_train,label='Generator (Train)')
plt.plot(dis_train,label='Discriminator (Train)')
plt.plot(dis_test,label='Discriminator (Test)')
plt.grid()
plt.legend()
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel(r'$L_c$')

In [ ]:
generated_images.shape, W.shape

In [ ]:
generated_images = generated_images.reshape(-1, 12000, 3, 1)

In [ ]:
X = np.concatenate([W,generated_images], axis=0)
X.shape

In [ ]:
Y.shape,sampled_labels.shape

In [ ]:
# Reshape sampled_labels to match Y's dimensions
sampled_labels_reshaped = sampled_labels.reshape(-1, 1)

# Now concatenate
y = np.concatenate([Y, sampled_labels_reshaped], axis=0)

# Check the shape
print(y.shape)

In [ ]:
seed = 42
np.random.seed(seed)
np.random.shuffle(X)
np.random.seed(seed)
np.random.shuffle(y)

In [ ]:
W_test = np.load('/kaggle/input/dataset-test/Test_Dataset_waveforms_train.npy')
Y_test = np.load('/kaggle/input/dataset-test/Test_Dataset_labels_train.npy')
W_test.shape, Y_test.shape

In [ ]:
W_test = W_test.reshape(-1,1600,3,1)

In [ ]:
def identity_block(x, filter):
    # copy tensor to variable called x_skip
    x_skip = x
    # Layer 1
    x = tf.keras.layers.Conv2D(filter, (3,3), padding = 'same')(x)
    x = tf.keras.layers.BatchNormalization(axis=3)(x)
    x = tf.keras.layers.Activation('relu')(x)
    # Layer 2
    x = tf.keras.layers.Conv2D(filter, (3,3), padding = 'same')(x)
    x = tf.keras.layers.BatchNormalization(axis=3)(x)
    # Add Residue
    x = tf.keras.layers.Add()([x, x_skip])
    x = tf.keras.layers.Activation('relu')(x)
    return x

def convolutional_block(x, filter):
    # copy tensor to variable called x_skip
    x_skip = x
    # Layer 1
    x = tf.keras.layers.Conv2D(filter, (3,3), padding = 'same', strides = (2,2))(x)
    x = tf.keras.layers.BatchNormalization(axis=3)(x)
    x = tf.keras.layers.Activation('relu')(x)
    # Layer 2
    x = tf.keras.layers.Conv2D(filter, (3,3), padding = 'same')(x)
    x = tf.keras.layers.BatchNormalization(axis=3)(x)
    x = tf.keras.layers.Activation('relu')(x)
    # Processing Residue with conv(1,1)
    x_skip = tf.keras.layers.Conv2D(filter, (1,1), strides = (2,2))(x_skip)
    # Add Residue
    x = tf.keras.layers.Add()([x, x_skip])
    x = tf.keras.layers.Activation('relu')(x)
    return x

def ResNet34(shape = (1600, 3, 1), classes = 2):
    # Step 1 (Setup Input Layer)
    x_input = tf.keras.layers.Input(shape)
    x = tf.keras.layers.ZeroPadding2D((3, 3))(x_input)
    # Step 2 (Initial Conv layer along with maxPool)
    x = tf.keras.layers.Conv2D(64, kernel_size=5, padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.MaxPool2D(pool_size=2, padding='same')(x)
    # Define size of sub-blocks and initial filter size
    block_layers = [3,3,4]
    filter_size = 64
    # Step 3 Add the Resnet Blocks
    for i in range(2):
        if i == 0:
            # For sub-block 1 Residual/Convolutional block not needed
            for j in range(block_layers[i]):
                x = identity_block(x, filter_size)
        else:
            # One Residual/Convolutional Block followed by Identity blocks
            # The filter size will go on increasing by a factor of 2
            filter_size = filter_size*2
            x = convolutional_block(x, filter_size)
            for j in range(block_layers[i] - 1):
                x = identity_block(x, filter_size)
    # Step 4 End Dense Network
    x = tf.keras.layers.AveragePooling2D((2,2), padding = 'same')(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(512, activation = 'relu')(x)
    x = tf.keras.layers.Dense(1, activation = 'softmax')(x)
    model = tf.keras.models.Model(inputs = x_input, outputs = x, name = "ResNet34")
    return model

In [ ]:
model1 = ResNet34()
model1.compile(optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
                      loss='binary_crossentropy',
             metrics=[tf.keras.metrics.Precision()])
model1.fit(X,y,epochs=1, batch_size=32, verbose=True)

In [ ]:
model2 = ResNet34()
model2.compile(optimizer=Adam(learning_rate=adam_lr, beta_1=adam_beta_1),
                      loss='binary_crossentropy',
             metrics=[tf.keras.metrics.Precision()])
model2.fit(W,Y,epochs=1, batch_size=32, verbose=True)